# CS781 Phase 6 — Experiment 6: Model 3 + Pfeiffer Adapters

Compact exploratory run (~1 hour target). Uses the exact Phase-1 Model-3 encoder–decoder architecture and LegalBPE tokenizer. All outputs stay directly in one main folder.

In [2]:
import os,re,gc,json,math,time,random,zipfile,glob
from pathlib import Path
from collections import Counter
import numpy as np, torch
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset,DataLoader
from huggingface_hub import hf_hub_download
SEED=2026; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type=="cuda": torch.backends.cuda.matmul.allow_tf32=True
try:
 from kaggle_secrets import UserSecretsClient
 HF_TOKEN=UserSecretsClient().get_secret("HF_TOKEN")
except Exception: HF_TOKEN=None
REPO="Exploration-Lab/CS-781-Capstone"
ROOT=Path("/kaggle/working/phase6_exp6_model3_pfeiffer"); ROOT.mkdir(parents=True,exist_ok=True)
ADAPTER_PATH=ROOT/"model3_pfeiffer_best.pt"; PRED_PATH=ROOT/"prediction.json"; ZIP_PATH=ROOT/"submission.zip"
D_MODEL=512; N_ENCODER_LAYERS=9; N_DECODER_LAYERS=9; N_HEADS=8; FFN_DIM=2048; DROPOUT=0.0
MAX_ENCODER_POSITIONS=2048; MAX_DECODER_POSITIONS=1536; ADAPTER_BOTTLENECK=64
N_TRAIN_LIMIT=900; N_VAL_LIMIT=64; EPOCHS=1; MAX_SOURCE_TOKENS=768; MAX_TARGET_TOKENS=384
MICRO_BATCH=1; GRAD_ACCUM=8; LR=2e-4; WEIGHT_DECAY=0.01; LOG_EVERY=25; VAL_EVERY=100; MAX_GEN=384
INPUT="/kaggle/input"
def find_file(pattern):
 hits=glob.glob(os.path.join(INPUT,"**",pattern),recursive=True)
 if not hits: raise FileNotFoundError(f"Missing {pattern} under /kaggle/input")
 return hits[0]
CKPT_PATH=find_file("encdec_full_best*.pt"); TOKENIZER_PATH=find_file("tokenizer (1).json")
print("GPU",torch.cuda.get_device_name(0) if DEVICE.type=="cuda" else "CPU")
print("Checkpoint",CKPT_PATH); print("Tokenizer",TOKENIZER_PATH)


GPU Tesla T4
Checkpoint /kaggle/input/datasets/aaryant23/phase2-model3-ckpts/encdec_full_best (2).pt
Tokenizer /kaggle/input/datasets/aaryant23/phase2-model3-ckpts/tokenizer (1).json


In [3]:
from collections import Counter
BPE_SPECIALS = ["<pad>", "<unk>", "<s>", "</s>"]
# Cell 3 — Custom byte-level BPE (same assignment design, scaled up)
BPE_SPECIALS = ["<pad>", "<unk>", "<s>", "</s>"]

class LegalBPE:
    def __init__(self, vocab_size=5000):
        self.vocab_size = int(vocab_size)
        self.merges = []
        self.itos = list(BPE_SPECIALS)
        self.stoi = {t:i for i,t in enumerate(self.itos)}
        self.symbol_bytes = {}
        for b in range(256):
            s = f"{b:02x}"
            self.symbol_bytes[s] = bytes([b])
            self.stoi[s] = len(self.itos)
            self.itos.append(s)
        self._reset_encode_cache()

    @property
    def pad_token_id(self): return self.stoi["<pad>"]
    @property
    def unk_token_id(self): return self.stoi["<unk>"]
    @property
    def bos_token_id(self): return self.stoi["<s>"]
    @property
    def eos_token_id(self): return self.stoi["</s>"]
    def __len__(self): return len(self.itos)

    @staticmethod
    def word_to_symbols(word):
        return [f"{b:02x}" for b in word.encode("utf-8")]

    def _reset_encode_cache(self):
        self._ranks = {(l,r): i for i,(l,r) in enumerate(self.merges)}
        self._word_cache = {}
        self._cache_nmerges = len(self.merges)

    def fit(self, sentences, min_count=2):
        word_counts = Counter()
        seen_chars = 0
        t0 = time.time()

        for sentence in sentences:
            seen_chars += len(sentence)
            word_counts.update(re.findall(r"\S+|\s+", sentence))
            if seen_chars and seen_chars % 10_000_000 < len(sentence):
                print(f"  BPE corpus read: {seen_chars/1e6:.0f}M chars | unique units={len(word_counts):,}", flush=True)

        if min_count > 1:
            word_counts = Counter({w:c for w,c in word_counts.items() if c >= min_count})

        print(f"BPE training units after min_count={min_count}: {len(word_counts):,}")
        print(f"Counting time: {(time.time()-t0)/60:.1f} min")

        words = list(word_counts)
        freqs = [word_counts[w] for w in words]
        syms = [self.word_to_symbols(w) for w in words]

        target_merges = self.vocab_size - len(self.itos)
        pair_counts = Counter()
        pair_words = {}

        for wi, s in enumerate(syms):
            f = freqs[wi]
            for j in range(len(s)-1):
                p = (s[j], s[j+1])
                pair_counts[p] += f
                pair_words.setdefault(p, set()).add(wi)

        heap = [(-c,p) for p,c in pair_counts.items()]
        heapq.heapify(heap)

        for merge_num in range(target_merges):
            best_pair = None
            best_freq = 0
            while heap:
                negc, p = heap[0]
                if pair_counts.get(p,0) == -negc and -negc > 0:
                    best_pair, best_freq = p, -negc
                    break
                heapq.heappop(heap)

            if best_pair is None or best_freq <= 1:
                print("No useful pairs remain.")
                break

            left, right = best_pair
            new_symbol = left + right
            self.merges.append([left,right])
            self.symbol_bytes[new_symbol] = self.symbol_bytes[left] + self.symbol_bytes[right]
            if new_symbol not in self.stoi:
                self.stoi[new_symbol] = len(self.itos)
                self.itos.append(new_symbol)

            touched = set()
            for wi in list(pair_words.get(best_pair, ())):
                old = syms[wi]
                f = freqs[wi]

                # Remove old local pair contributions.
                for j in range(len(old)-1):
                    p = (old[j], old[j+1])
                    pair_counts[p] -= f
                    touched.add(p)

                merged = []
                j = 0
                while j < len(old):
                    if j+1 < len(old) and old[j] == left and old[j+1] == right:
                        merged.append(new_symbol)
                        j += 2
                    else:
                        merged.append(old[j])
                        j += 1
                syms[wi] = merged

                # Add new local pair contributions.
                for j in range(len(merged)-1):
                    p = (merged[j], merged[j+1])
                    pair_counts[p] += f
                    pair_words.setdefault(p, set()).add(wi)
                    touched.add(p)

            pair_words.pop(best_pair, None)

            for p in touched:
                c = pair_counts.get(p,0)
                if c > 0:
                    heapq.heappush(heap, (-c,p))
                else:
                    pair_counts.pop(p, None)

            if (merge_num+1) % 1000 == 0 or merge_num == 0 or merge_num+1 == target_merges:
                print(
                    f"merge {merge_num+1:,}/{target_merges:,} | "
                    f"freq={best_freq:,} | vocab={len(self.itos):,}",
                    flush=True
                )

        self.vocab_size = len(self.itos)
        self._reset_encode_cache()
        print(f"BPE complete: vocab={len(self):,}")
        return self

    def encode_word(self, word):
        if self._cache_nmerges != len(self.merges):
            self._reset_encode_cache()

        hit = self._word_cache.get(word)
        if hit is not None:
            return list(hit)

        symbols = self.word_to_symbols(word)
        ranks = self._ranks

        while len(symbols) > 1:
            best = None
            best_rank = None
            for j in range(len(symbols)-1):
                r = ranks.get((symbols[j], symbols[j+1]))
                if r is not None and (best_rank is None or r < best_rank):
                    best = (symbols[j], symbols[j+1])
                    best_rank = r
            if best is None:
                break

            left, right = best
            merged = []
            j = 0
            while j < len(symbols):
                if j+1 < len(symbols) and symbols[j] == left and symbols[j+1] == right:
                    merged.append(left+right)
                    j += 2
                else:
                    merged.append(symbols[j])
                    j += 1
            symbols = merged

        # Bound cache memory for 500M-token preprocessing.
        if len(self._word_cache) >= 250_000:
            self._word_cache.clear()
        self._word_cache[word] = tuple(symbols)
        return symbols

    def encode(self, text):
        out = []
        unk = self.unk_token_id
        for unit in re.findall(r"\S+|\s+", text):
            for s in self.encode_word(unit):
                out.append(self.stoi.get(s, unk))
        return out

    def decode(self, ids):
        buf = bytearray()
        for idx in ids:
            idx = int(idx)
            if idx < 0 or idx >= len(self.itos):
                raise ValueError(f"invalid token id {idx}")
            tok = self.itos[idx]
            if tok in BPE_SPECIALS:
                continue
            buf.extend(self.symbol_bytes[tok])
        return bytes(buf).decode("utf-8", errors="replace")

    def save_json(self, path):
        with open(path, "w", encoding="utf-8") as f:
            json.dump(
                {"vocab_size":len(self), "merges":self.merges, "itos":self.itos},
                f, ensure_ascii=False
            )

    @classmethod
    def load_json(cls, path):
        obj = json.load(open(path, encoding="utf-8"))
        tok = cls(obj["vocab_size"])
        tok.merges = [list(x) for x in obj["merges"]]
        tok.itos = list(obj["itos"])
        tok.stoi = {t:i for i,t in enumerate(tok.itos)}
        tok.vocab_size = len(tok.itos)

        tok.symbol_bytes = {}
        for b in range(256):
            s=f"{b:02x}"
            tok.symbol_bytes[s]=bytes([b])
        for l,r in tok.merges:
            tok.symbol_bytes[l+r] = tok.symbol_bytes[l] + tok.symbol_bytes[r]

        tok._reset_encode_cache()
        return tok

print("Tokenizer class ready.")

Tokenizer class ready.


In [4]:

# Phase-6 legal summarization data
train_path=hf_hub_download(repo_id=REPO,filename="Phase_6_Summ/Train/train.json",repo_type="dataset",token=HF_TOKEN)
test_path=hf_hub_download(repo_id=REPO,filename="Phase_6_Summ/Test/test.json",repo_type="dataset",token=HF_TOKEN)
train_raw=json.load(open(train_path,encoding="utf-8")); test_raw=json.load(open(test_path,encoding="utf-8"))
train_raw=[r for r in train_raw if r.get("document","").strip() and r.get("summary","").strip()]
random.Random(SEED).shuffle(train_raw); train_raw=train_raw[:N_TRAIN_LIMIT+N_VAL_LIMIT]
val_rows=train_raw[:N_VAL_LIMIT]; train_rows=train_raw[N_VAL_LIMIT:]
BPE_SPECIALS=["<pad>","<unk>","<s>","</s>"]
tokenizer=LegalBPE.load_json(TOKENIZER_PATH)
PAD,UNK,BOS,EOS=tokenizer.pad_token_id,tokenizer.unk_token_id,tokenizer.bos_token_id,tokenizer.eos_token_id
print("Usable train",len(train_rows),"validation",len(val_rows),"test",len(test_raw),"vocab",len(tokenizer))
def enc_pair(r):
 s=tokenizer.encode(r["document"])[:MAX_SOURCE_TOKENS]; t=tokenizer.encode(r["summary"])[:MAX_TARGET_TOKENS]
 return (s or [UNK],t or [UNK])
train_pairs=[enc_pair(r) for r in train_rows]; val_pairs=[enc_pair(r) for r in val_rows]


Phase_6_Summ/Train/train.json: reconstructing file:   0%|          |  0.00B /  211MB            

Phase_6_Summ/Train/train.json: downloading bytes:           |  0.00B            

test.json:   0%|          | 0.00/2.73M [00:00<?, ?B/s]

Usable train 900 validation 64 test 100 vocab 24576


In [5]:
# Cell 9 — ~79M Encoder–Decoder Transformer (SDPA, cross-attention, decode KV cache)
# v2 changes vs v1:
#   * token embeddings are scaled by sqrt(d_model) before the sinusoidal positions are added
#     (v1 added unit-scale positions to 0.02-std embeddings => positions drowned the tokens)
#   * encoder positions are derived from the padding mask (so padded / doc-masked rows get positions 0..n-1)
#   * decoder self-attention handles "cache + several new tokens" correctly (needed for prompt priming)
#   * residual-branch init uses the true branch count (3 per decoder layer)
#   * build_model() so Cell 11 always starts from fresh weights (no more "rerun Cell 9" step)
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_positions):
        super().__init__()
        pos = torch.arange(max_positions, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32)
            * (-math.log(10000.0) / d_model)
        )
        pe = torch.zeros(max_positions, d_model, dtype=torch.float32)
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe, persistent=False)

    def forward(self, position_ids):
        return self.pe[position_ids]


class EncoderSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = dropout

    def forward(self, x, key_valid=None):
        B, T, C = x.shape
        qkv = self.qkv(x).view(B, T, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.unbind(dim=2)
        q, k, v = q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2)

        attn_mask = None
        if key_valid is not None:
            attn_mask = key_valid[:, None, None, :]  # True = valid encoder key

        y = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=attn_mask,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=False,
        )
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.out_proj(y)


class DecoderSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = dropout

    def forward(self, x, past_kv=None, use_cache=False):
        B, T, C = x.shape
        qkv = self.qkv(x).view(B, T, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.unbind(dim=2)
        q, k, v = q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2)

        mask, is_causal = None, True
        if past_kv is not None:
            pk, pv = past_kv
            k = torch.cat([pk, k], dim=2)
            v = torch.cat([pv, v], dim=2)
            if T == 1:
                is_causal = False                      # the new token may see every cached key
            else:
                S = k.size(2)
                P = S - T                              # query i sits at global position P+i
                mask = torch.ones(T, S, device=x.device).tril(diagonal=P).bool()
                is_causal = False

        y = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=mask,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=is_causal,
        )
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.out_proj(y)
        return y, ((k, v) if use_cache else None)


class CrossAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.q_proj = nn.Linear(d_model, d_model)
        self.kv_proj = nn.Linear(d_model, 2 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = dropout

    def forward(self, x, memory, memory_valid=None, cached_kv=None, use_cache=False):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        if cached_kv is None:
            S = memory.size(1)
            kv = self.kv_proj(memory).view(B, S, 2, self.n_heads, self.head_dim)
            k, v = kv.unbind(dim=2)
            k, v = k.transpose(1, 2), v.transpose(1, 2)
        else:
            k, v = cached_kv

        attn_mask = None
        if memory_valid is not None:
            attn_mask = memory_valid[:, None, None, :]

        y = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=attn_mask,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=False,
        )
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.out_proj(y)
        return y, ((k, v) if use_cache else None)



class PfeifferAdapter(nn.Module):
    def __init__(self,d_model,bottleneck=64):
        super().__init__(); self.down=nn.Linear(d_model,bottleneck); self.act=nn.GELU(); self.up=nn.Linear(bottleneck,d_model)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)
    def forward(self,x): return self.up(self.act(self.down(x)))

class EncoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, ffn_dim, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = EncoderSelfAttention(d_model, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, ffn_dim),
            nn.GELU(),
            nn.Linear(ffn_dim, d_model),
            nn.Dropout(dropout),
        )
        self.drop = nn.Dropout(dropout)
        self.adapter = PfeifferAdapter(d_model, ADAPTER_BOTTLENECK)

    def forward(self, x, key_valid=None):
        x = x + self.drop(self.attn(self.ln1(x), key_valid=key_valid))
        x = x + self.ff(self.ln2(x))
        x = x + self.adapter(x)
        return x


class DecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, ffn_dim, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.self_attn = DecoderSelfAttention(d_model, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.cross_attn = CrossAttention(d_model, n_heads, dropout)
        self.ln3 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, ffn_dim),
            nn.GELU(),
            nn.Linear(ffn_dim, d_model),
            nn.Dropout(dropout),
        )
        self.drop = nn.Dropout(dropout)
        self.adapter = PfeifferAdapter(d_model, ADAPTER_BOTTLENECK)

    def forward(
        self, x, memory, memory_valid=None,
        past_self_kv=None, cached_cross_kv=None, use_cache=False
    ):
        a, new_self = self.self_attn(
            self.ln1(x), past_kv=past_self_kv, use_cache=use_cache
        )
        x = x + self.drop(a)

        c, new_cross = self.cross_attn(
            self.ln2(x),
            memory,
            memory_valid=memory_valid,
            cached_kv=cached_cross_kv,
            use_cache=use_cache,
        )
        x = x + self.drop(c)
        x = x + self.ff(self.ln3(x))
        x = x + self.adapter(x)
        return x, new_self, new_cross


class EncoderDecoderLM(nn.Module):
    def __init__(
        self, vocab_size, d_model, n_encoder_layers, n_decoder_layers,
        n_heads, ffn_dim, dropout, max_encoder_positions, max_decoder_positions
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.emb_scale = math.sqrt(d_model)
        self.max_encoder_positions = max_encoder_positions
        self.max_decoder_positions = max_decoder_positions

        # Shared embedding across encoder input, decoder input and output projection.
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.enc_pos = SinusoidalPositionalEncoding(d_model, max_encoder_positions)
        self.dec_pos = SinusoidalPositionalEncoding(d_model, max_decoder_positions)
        self.drop = nn.Dropout(dropout)

        self.encoder = nn.ModuleList([
            EncoderBlock(d_model, n_heads, ffn_dim, dropout)
            for _ in range(n_encoder_layers)
        ])
        self.decoder = nn.ModuleList([
            DecoderBlock(d_model, n_heads, ffn_dim, dropout)
            for _ in range(n_decoder_layers)
        ])
        self.enc_ln = nn.LayerNorm(d_model)
        self.dec_ln = nn.LayerNorm(d_model)

        self.apply(self._init_weights)

        # GPT-2 style depth scaling: std = 0.02 / sqrt(#residual branches)
        enc_scaled = 0.02 / math.sqrt(2 * n_encoder_layers)   # attn + ffn
        dec_scaled = 0.02 / math.sqrt(3 * n_decoder_layers)   # self-attn + cross-attn + ffn
        for block in self.encoder:
            nn.init.normal_(block.attn.out_proj.weight, mean=0.0, std=enc_scaled)
            nn.init.normal_(block.ff[2].weight, mean=0.0, std=enc_scaled)
        for block in self.decoder:
            nn.init.normal_(block.self_attn.out_proj.weight, mean=0.0, std=dec_scaled)
            nn.init.normal_(block.cross_attn.out_proj.weight, mean=0.0, std=dec_scaled)
            nn.init.normal_(block.ff[2].weight, mean=0.0, std=dec_scaled)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def encode(self, input_ids, key_valid=None, position_ids=None):
        B, S = input_ids.shape
        if S > self.max_encoder_positions:
            raise ValueError(f"encoder length {S} > {self.max_encoder_positions}")

        if position_ids is None:
            if key_valid is None:
                position_ids = torch.arange(S, device=input_ids.device)[None, :].expand(B, S)
            else:
                # valid tokens always get positions 0..n-1, wherever the padding / masked tokens sit
                position_ids = (key_valid.long().cumsum(1) - 1).clamp(min=0)

        x = self.token_emb(input_ids) * self.emb_scale \
            + self.enc_pos(position_ids).to(self.token_emb.weight.dtype)
        x = self.drop(x)
        for block in self.encoder:
            x = block(x, key_valid=key_valid)
        return self.enc_ln(x)

    def decode_hidden(
        self, decoder_ids, memory, memory_valid=None,
        past_self_key_values=None, cross_key_values=None,
        use_cache=False, position_ids=None
    ):
        B, T = decoder_ids.shape
        past_len = 0
        if past_self_key_values is not None:
            past_len = past_self_key_values[0][0].size(2)

        if past_len + T > self.max_decoder_positions:
            raise ValueError(
                f"decoder length {past_len + T} > {self.max_decoder_positions}"
            )

        if position_ids is None:
            position_ids = torch.arange(
                past_len, past_len + T, device=decoder_ids.device
            )[None, :].expand(B, T)

        x = self.token_emb(decoder_ids) * self.emb_scale \
            + self.dec_pos(position_ids).to(self.token_emb.weight.dtype)
        x = self.drop(x)

        new_self, new_cross = ([] if use_cache else None), ([] if use_cache else None)

        for i, block in enumerate(self.decoder):
            ps = None if past_self_key_values is None else past_self_key_values[i]
            ck = None if cross_key_values is None else cross_key_values[i]
            x, s_kv, c_kv = block(
                x, memory, memory_valid=memory_valid,
                past_self_kv=ps, cached_cross_kv=ck, use_cache=use_cache
            )
            if use_cache:
                new_self.append(s_kv)
                new_cross.append(c_kv)

        return self.dec_ln(x), new_self, new_cross

    def project(self, hidden):
        return F.linear(hidden, self.token_emb.weight)


def build_model():
    return EncoderDecoderLM(
        vocab_size=len(tokenizer),
        d_model=D_MODEL,
        n_encoder_layers=N_ENCODER_LAYERS,
        n_decoder_layers=N_DECODER_LAYERS,
        n_heads=N_HEADS,
        ffn_dim=FFN_DIM,
        dropout=DROPOUT,
        max_encoder_positions=MAX_ENCODER_POSITIONS,
        max_decoder_positions=MAX_DECODER_POSITIONS,
    ).to(DEVICE)


In [6]:

model=build_model()
obj=torch.load(CKPT_PATH,map_location="cpu",weights_only=False); state=obj["model"] if "model" in obj else obj
missing,unexpected=model.load_state_dict(state,strict=False)
other=[k for k in missing if ".adapter." not in k]
print("Missing adapter keys",len(missing)-len(other),"other missing",other[:5],"unexpected",unexpected[:5])
assert not other and not unexpected
for p in model.parameters(): p.requires_grad=False
for n,p in model.named_parameters():
 if ".adapter." in n: p.requires_grad=True
print("Trainable",sum(p.numel() for p in model.parameters() if p.requires_grad),"of",sum(p.numel() for p in model.parameters()))
model.to(DEVICE)


Missing adapter keys 72 other missing [] unexpected []
Trainable 1190016 of 79982720


EncoderDecoderLM(
  (token_emb): Embedding(24576, 512)
  (enc_pos): SinusoidalPositionalEncoding()
  (dec_pos): SinusoidalPositionalEncoding()
  (drop): Dropout(p=0.0, inplace=False)
  (encoder): ModuleList(
    (0-8): 9 x EncoderBlock(
      (ln1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (attn): EncoderSelfAttention(
        (qkv): Linear(in_features=512, out_features=1536, bias=True)
        (out_proj): Linear(in_features=512, out_features=512, bias=True)
      )
      (ln2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (ff): Sequential(
        (0): Linear(in_features=512, out_features=2048, bias=True)
        (1): GELU(approximate='none')
        (2): Linear(in_features=2048, out_features=512, bias=True)
        (3): Dropout(p=0.0, inplace=False)
      )
      (drop): Dropout(p=0.0, inplace=False)
      (adapter): PfeifferAdapter(
        (down): Linear(in_features=512, out_features=64, bias=True)
        (act): GELU(approximate='none')
        (u

In [7]:

class DS(Dataset):
 def __init__(self,p): self.p=p
 def __len__(self): return len(self.p)
 def __getitem__(self,i): return self.p[i]
def collate(batch):
 B=len(batch); S=max(len(x[0]) for x in batch); T=max(len(x[1]) for x in batch)+1
 e=torch.full((B,S),PAD,dtype=torch.long); mask=torch.zeros((B,S),dtype=torch.bool)
 d=torch.full((B,T),PAD,dtype=torch.long); y=torch.full((B,T),-100,dtype=torch.long)
 for i,(s,t) in enumerate(batch):
  e[i,:len(s)]=torch.tensor(s); mask[i,:len(s)]=True
  seq=t+[EOS]; di=[BOS]+seq[:-1]; d[i,:len(di)]=torch.tensor(di); y[i,:len(seq)]=torch.tensor(seq)
 return e.to(DEVICE),mask.to(DEVICE),d.to(DEVICE),y.to(DEVICE)
tr=DataLoader(DS(train_pairs),batch_size=MICRO_BATCH,shuffle=True,collate_fn=collate)
va=DataLoader(DS(val_pairs),batch_size=1,shuffle=False,collate_fn=collate)
def loss_fn(b):
 e,mask,d,y=b; mem=model.encode(e,key_valid=mask); h,_,_=model.decode_hidden(d,mem,memory_valid=mask)
 z=model.project(h).float(); return F.cross_entropy(z.reshape(-1,z.size(-1)),y.reshape(-1),ignore_index=-100)
@torch.no_grad()
def evaluate():
 model.eval(); vals=[]
 for b in va:
  with torch.amp.autocast("cuda",dtype=torch.float16,enabled=DEVICE.type=="cuda"): vals.append(loss_fn(b).item())
 model.train(); return float(np.mean(vals))
opt=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=LR,weight_decay=WEIGHT_DECAY)
steps=math.ceil(len(tr)/GRAD_ACCUM); total=steps*EPOCHS; warm=max(1,int(.05*total)); scaler=torch.amp.GradScaler("cuda",enabled=DEVICE.type=="cuda")
best=float("inf"); hist=[]; step=0; start=time.time(); model.train(); opt.zero_grad(set_to_none=True)
print(f"Training {len(train_rows)} examples | {total} optimizer steps | {EPOCHS} epoch")
for ep in range(EPOCHS):
 for bi,b in enumerate(tr,1):
  with torch.amp.autocast("cuda",dtype=torch.float16,enabled=DEVICE.type=="cuda"): l=loss_fn(b)/GRAD_ACCUM
  scaler.scale(l).backward()
  if bi%GRAD_ACCUM==0 or bi==len(tr):
   scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad],1.0)
   scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); step+=1
   if step<=warm:
    for g in opt.param_groups:g["lr"]=LR*step/warm
   if step%LOG_EVERY==0 or step==1:
    el=time.time()-start; eta=(total-step)*el/max(step,1)
    print(f"step {step}/{total} | loss {l.item()*GRAD_ACCUM:.4f} | elapsed {el/60:.1f}m | ETA {eta/60:.1f}m",flush=True)
   if step%VAL_EVERY==0 or step==total:
    v=evaluate(); hist.append({"step":step,"val_loss":v}); print("validation loss",step,v,flush=True)
    if v<best:
     best=v; torch.save({"adapter":{k:v.detach().cpu() for k,v in model.state_dict().items() if ".adapter." in k},"step":step,"val_loss":v},ADAPTER_PATH)
    model.train()
if not ADAPTER_PATH.exists(): torch.save({"adapter":{k:v.detach().cpu() for k,v in model.state_dict().items() if ".adapter." in k},"step":step,"val_loss":best},ADAPTER_PATH)
json.dump(hist,open(ROOT/"training_history.json","w"),indent=2)
print(f"Training done in {(time.time()-start)/60:.1f}m | best val {best:.5f}")


Training 900 examples | 113 optimizer steps | 1 epoch
step 1/113 | loss 1.9410 | elapsed 0.0m | ETA 2.9m
step 25/113 | loss 1.8488 | elapsed 0.2m | ETA 0.6m
step 50/113 | loss 1.8753 | elapsed 0.3m | ETA 0.4m
step 75/113 | loss 2.2903 | elapsed 0.4m | ETA 0.2m
step 100/113 | loss 1.6896 | elapsed 0.6m | ETA 0.1m
validation loss 100 1.9544011037796736
validation loss 113 1.9499975088983774
Training done in 0.7m | best val 1.95000


In [9]:
%pip install -q bert-score rouge-score

Note: you may need to restart the kernel to use updated packages.


In [10]:

# Load best adapter; cached encoder-decoder greedy generation with repetition controls.
model.load_state_dict(torch.load(ADAPTER_PATH,map_location="cpu",weights_only=False)["adapter"],strict=False); model.eval()
@torch.inference_mode()
def generate_one(document,max_new=MAX_GEN,min_new=40,ng=8,rep=1.05):
 s=tokenizer.encode(document)[:MAX_SOURCE_TOKENS] or [UNK]; e=torch.tensor([s],device=DEVICE); valid=torch.ones_like(e,dtype=torch.bool)
 with torch.amp.autocast("cuda",dtype=torch.float16,enabled=DEVICE.type=="cuda"):
  mem=model.encode(e,key_valid=valid); d=torch.tensor([[BOS]],device=DEVICE); h,sk,ck=model.decode_hidden(d,mem,memory_valid=valid,use_cache=True)
 logits=model.project(h[:,-1:])[:,-1,:].float(); out=[]
 for st in range(max_new):
  z=logits.clone(); z[:,[PAD,UNK,BOS]]=-float("inf")
  if st<min_new:z[:,EOS]=-float("inf")
  if rep!=1 and out:
   ids=torch.tensor(list(set(out)),device=DEVICE); v=z[:,ids]; z[:,ids]=torch.where(v<0,v*rep,v/rep)
  if ng>=2 and len(out)>=ng-1:
   pref=tuple(out[-ng+1:]); ban=[out[j+ng-1] for j in range(len(out)-ng+1) if tuple(out[j:j+ng-1])==pref]
   if ban:z[:,list(set(ban))]=-float("inf")
  nxt=int(z.argmax(-1).item())
  if nxt==EOS and st>=min_new:break
  out.append(nxt); pos=torch.tensor([[st+1]],device=DEVICE)
  with torch.amp.autocast("cuda",dtype=torch.float16,enabled=DEVICE.type=="cuda"):
   h,sk,ck=model.decode_hidden(torch.tensor([[nxt]],device=DEVICE),mem,memory_valid=valid,past_self_key_values=sk,cross_key_values=ck,use_cache=True,position_ids=pos)
   logits=model.project(h[:,-1:])[:,-1,:].float()
 return tokenizer.decode(out).strip()
from bert_score import BERTScorer
scorer=BERTScorer(lang="en",batch_size=4,device=str(DEVICE),rescale_with_baseline=False)
from rouge_score import rouge_scorer
rs=rouge_scorer.RougeScorer(["rouge1","rougeL"],use_stemmer=True)
def score(rows,preds):
 refs=[r["summary"] for r in rows]; _,_,f=scorer.score(preds,refs)
 rr=[rs.score(r,p) for p,r in zip(preds,refs)]
 return {"BERTScore_F1":float(f.mean()),"ROUGE1_F1":float(np.mean([x["rouge1"].fmeasure for x in rr])),"ROUGEL_F1":float(np.mean([x["rougeL"].fmeasure for x in rr]))}
tune=val_rows[:16]; grid=[{"name":"greedy","ng":0,"rep":1.0,"min":40},{"name":"ng8_rep105","ng":8,"rep":1.05,"min":40}]
results=[]
for ci,cfg in enumerate(grid,1):
 print(f"\nDecode {ci}/{len(grid)} {cfg}",flush=True); pp=[]; t0=time.time()
 for i,r in enumerate(tune,1):
  pp.append(generate_one(r["document"],ng=cfg["ng"],rep=cfg["rep"],min_new=cfg["min"]))
  if i%4==0 or i==len(tune): print(f"[decode] {i}/{len(tune)} | elapsed {(time.time()-t0)/60:.1f}m | ETA {(time.time()-t0)*(len(tune)-i)/i/60:.1f}m",flush=True)
 sc=score(tune,pp); results.append({**cfg,**sc}); print("RESULT",results[-1],flush=True)
json.dump(results,open(ROOT/"decode_sweep.json","w"),indent=2); bestcfg=max(results,key=lambda x:x["BERTScore_F1"]); print("BEST",bestcfg)
final=val_rows[16:32]; vp=[]
for i,r in enumerate(final,1):
 vp.append(generate_one(r["document"],ng=bestcfg["ng"],rep=bestcfg["rep"],min_new=bestcfg["min"]))
 if i%4==0 or i==len(final):print(f"[validation] {i}/{len(final)}",flush=True)
print("FINAL VALIDATION",score(final,vp)); json.dump([{"id":r["id"],"summary":p} for r,p in zip(final,vp)],open(ROOT/"validation_predictions.json","w"),ensure_ascii=False,indent=2)
test_preds=[]; t0=time.time()
for i,r in enumerate(test_raw,1):
 p=generate_one(r["document"],ng=bestcfg["ng"],rep=bestcfg["rep"],min_new=bestcfg["min"]); test_preds.append({"id":r["id"],"summary":p})
 if i%5==0 or i==len(test_raw):
  json.dump(test_preds,open(ROOT/"prediction_partial.json","w"),ensure_ascii=False)
  print(f"[test] {i}/{len(test_raw)} | elapsed {(time.time()-t0)/60:.1f}m",flush=True)
json.dump(test_preds,open(PRED_PATH,"w"),ensure_ascii=False,indent=2)
with zipfile.ZipFile(ZIP_PATH,"w",zipfile.ZIP_DEFLATED) as z:z.write(PRED_PATH,arcname="prediction.json")
print("SUBMISSION",ZIP_PATH,"bytes",ZIP_PATH.stat().st_size)


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Decode 1/2 {'name': 'greedy', 'ng': 0, 'rep': 1.0, 'min': 40}
[decode] 4/16 | elapsed 0.2m | ETA 0.7m
[decode] 8/16 | elapsed 0.5m | ETA 0.5m
[decode] 12/16 | elapsed 0.7m | ETA 0.2m
[decode] 16/16 | elapsed 1.0m | ETA 0.0m
RESULT {'name': 'greedy', 'ng': 0, 'rep': 1.0, 'min': 40, 'BERTScore_F1': 0.6966249942779541, 'ROUGE1_F1': 0.02027185329206077, 'ROUGEL_F1': 0.020169729109054235}

Decode 2/2 {'name': 'ng8_rep105', 'ng': 8, 'rep': 1.05, 'min': 40}
[decode] 4/16 | elapsed 0.2m | ETA 0.5m
[decode] 8/16 | elapsed 0.4m | ETA 0.4m
[decode] 12/16 | elapsed 0.5m | ETA 0.2m
[decode] 16/16 | elapsed 0.7m | ETA 0.0m
RESULT {'name': 'ng8_rep105', 'ng': 8, 'rep': 1.05, 'min': 40, 'BERTScore_F1': 0.7838926315307617, 'ROUGE1_F1': 0.2140439276225003, 'ROUGEL_F1': 0.14502545043817194}
BEST {'name': 'ng8_rep105', 'ng': 8, 'rep': 1.05, 'min': 40, 'BERTScore_F1': 0.7838926315307617, 'ROUGE1_F1': 0.2140439276225003, 'ROUGEL_F1': 0.14502545043817194}
[validation] 4/16
[validation] 8/16
[validation] 12/

## Run order
Run cells top to bottom. Training is 900 examples, 1 epoch, 64 held-out examples. Decoding tune: 16 docs; final validation: 16 disjoint docs; official test: all test docs. Progress and ETA are printed during training and generation. Outputs are flat in `/kaggle/working/phase6_exp6_model3_pfeiffer/`.